# Day 83: Covariate Balance, Entropy & KL Divergence

*Beginner to Ninja: Math for Causal Inference*

How different are treated and control units, before and after adjustment? Standardised differences, variance ratios and distributional checks measure balance; entropy and the Kullback–Leibler divergence measure how far one distribution, or one set of weights, is from another, the language of tomorrow's entropy balancing.

**Learning outcomes**

- Compute standardised mean differences and variance ratios, and read a balance table.
- Compute entropy and KL divergences for discrete and normal distributions, and prove that KL is non-negative and asymmetric.
- Measure how far a set of weights departs from uniform with KL and effective sample size, and relate the two.
- Diagnose balance problems that mean comparisons miss.

## Theory

### Balance diagnostics

For a covariate with means $\bar x_1, \bar x_0$ and SDs $s_1, s_0$ among treated and controls:

- **Standardised mean difference:** $SMD = (\bar x_1 - \bar x_0)/\sqrt{(s_1^2 + s_0^2)/2}$. Rule of thumb: $|SMD| < 0.1$. Unlike t-statistics, SMDs do not grow with sample size.
- **Variance ratio:** $s_1^2/s_0^2$, ideally between 0.5 and 2.
- **Distributions:** empirical CDFs, Kolmogorov–Smirnov statistics, quantile plots; balance on squares and interactions.

After weighting or matching, compute the same statistics with weights. Balance is checked on covariates, never on outcomes.

### Entropy

For a discrete distribution $p$, the **entropy** is $H(p) = -\sum_k p_k\ln p_k$: maximal ($\ln K$) for the uniform distribution, zero for a point mass.

### Kullback–Leibler divergence

$$
D_{KL}(P \,\|\, Q) = \sum_k p_k\ln\frac{p_k}{q_k} \quad \Big(\text{or } \int p(x)\ln\frac{p(x)}{q(x)}dx\Big).
$$

Properties: $D_{KL} \ge 0$ with equality iff $P = Q$ (Gibbs' inequality); not symmetric. For normals,

$$
D_{KL}\big(N(\mu_1, \sigma_1^2) \,\|\, N(\mu_2, \sigma_2^2)\big) = \ln\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac12.
$$

### Why KL appears in causal inference

- **Covariate shift:** $D_{KL}$ between treated and control covariate distributions measures how hard the comparison is; IPW weights are density ratios $p_1(x)/p_0(x)$ (up to constants), the same quantity inside KL.
- **Weight dispersion:** for normalised weights $w_i$ on $n$ units, $D_{KL}(w \,\|\, \text{uniform}) = \sum_i w_i\ln(nw_i) = \ln n - H(w)$. Zero for equal weights; large when few units carry the weight. It is bounded by $\ln(n/ESS)$, where $ESS = 1/\sum w_i^2$.
- **Entropy balancing** (Day 84) chooses weights as close to uniform in KL as possible while achieving exact balance.

## Part 1 · Warm-up

### Exercise 1 · A standardised difference

Mean age 40 (SD 10) among treated, 35 (SD 8) among controls. Compute the SMD.

<details><summary>Show solution</summary>

**Step 1.** Pooled SD: $\sqrt{(100 + 64)/2} = \sqrt{82} \approx 9.06$.

**Step 2.** $SMD = 5/9.06 \approx 0.55$.

**Answer:** large imbalance (above 0.1).

</details>

*Your work:*

### Exercise 2 · A variance ratio

In Exercise 1, compute the variance ratio.

<details><summary>Show solution</summary>

**Step 1.** $100/64 \approx 1.56$: within the 0.5–2 range.

</details>

*Your work:*

### Exercise 3 · Entropy

Compute the entropy of $(0.5, 0.25, 0.25)$ and compare with the uniform distribution on three points.

<details><summary>Show solution</summary>

**Step 1.** $H = -(0.5\ln 0.5 + 2 \times 0.25\ln 0.25) = 0.347 + 0.693 = 1.040$.

**Step 2.** Uniform: $\ln 3 \approx 1.099$.

</details>

*Your work:*

### Exercise 4 · KL between Bernoullis

Compute $D_{KL}(\text{Bern}(0.3) \,\|\, \text{Bern}(0.5))$.

<details><summary>Show solution</summary>

**Step 1.** $0.3\ln(0.3/0.5) + 0.7\ln(0.7/0.5) = 0.3\ln 0.6 + 0.7\ln 1.4$.

**Step 2.** $= -0.153 + 0.236 = 0.082$.

</details>

*Your work:*

### Exercise 5 · KL between normals

Compute $D_{KL}(N(1, 1) \,\|\, N(0, 1))$ and $D_{KL}(N(0, 4) \,\|\, N(0, 1))$.

<details><summary>Show solution</summary>

**Step 1.** First: $0 + (1 + 1)/2 - 1/2 = 0.5$.

**Step 2.** Second: $\ln(1/2) + 4/2 - 1/2 = -0.693 + 1.5 = 0.807$.

**Answer:** a doubling of the SD shifts the distribution "more" than a one-SD shift of the mean.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Gibbs' inequality

Prove $D_{KL}(P \,\|\, Q) \ge 0$ with equality iff $P = Q$.

<details><summary>Show solution</summary>

**Step 1.** $-D_{KL} = \sum_k p_k\ln(q_k/p_k)$.

**Step 2.** $\ln$ is concave; by Jensen, $\sum_k p_k\ln(q_k/p_k) \le \ln\sum_k p_k(q_k/p_k) = \ln\sum_k q_k \le \ln 1 = 0$.

**Step 3.** Equality in Jensen requires $q_k/p_k$ constant, so $P = Q$. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · KL between normals

Derive the formula for $D_{KL}(N(\mu_1, \sigma_1^2) \,\|\, N(\mu_2, \sigma_2^2))$.

<details><summary>Show solution</summary>

**Step 1.** $\ln\frac{p(x)}{q(x)} = \ln\frac{\sigma_2}{\sigma_1} - \frac{(x - \mu_1)^2}{2\sigma_1^2} + \frac{(x - \mu_2)^2}{2\sigma_2^2}$.

**Step 2.** Take expectations under $P$: $E_P[(x - \mu_1)^2] = \sigma_1^2$ and $E_P[(x - \mu_2)^2] = \sigma_1^2 + (\mu_1 - \mu_2)^2$.

**Step 3.** $D_{KL} = \ln\frac{\sigma_2}{\sigma_1} - \frac12 + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2}$. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · Asymmetry

Using Exercise 7, show $D_{KL}(N(0, 4) \,\|\, N(0, 1)) \ne D_{KL}(N(0, 1) \,\|\, N(0, 4))$.

<details><summary>Show solution</summary>

**Step 1.** First: 0.807 (Exercise 5).

**Step 2.** Second: $\ln 2 + 1/8 - 1/2 = 0.693 - 0.375 = 0.318$.

**Answer:** different: KL is not a distance. $\blacksquare$

</details>

*Your work:*

### Exercise 9 · Weights, entropy and ESS

For normalised weights $w$ on $n$ units, prove $D_{KL}(w \,\|\, u) = \ln n - H(w)$ and $D_{KL}(w \,\|\, u) \le \ln(n/ESS)$.

<details><summary>Show solution</summary>

**Step 1.** $\sum_i w_i\ln(w_i/(1/n)) = \ln n + \sum_i w_i\ln w_i = \ln n - H(w)$.

**Step 2.** Jensen: $\sum_i w_i\ln(nw_i) \le \ln\sum_i w_i(nw_i) = \ln(n\sum_i w_i^2) = \ln(n/ESS)$. $\blacksquare$

</details>

*Your work:*

### Exercise 10 · Mean balance is not distributional balance

Give two distributions with the same mean and SMD 0 whose KL divergence is large, and explain why this matters for outcome models.

<details><summary>Show solution</summary>

**Step 1.** $N(0, 1)$ and $N(0, 9)$: same mean, SMD 0; $D_{KL}(N(0, 9) \,\|\, N(0, 1)) = \ln(1/3) + 9/2 - 1/2 \approx 2.90$.

**Step 2.** If the outcome depends on $x^2$ (for example, health risk rising with distance from normal BMI), the groups differ in expected outcomes despite equal means.

**Answer:** also check variance ratios, higher moments and distributions. $\blacksquare$

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · A balance table

In a study of health insurance and preventive care: age 45 (SD 12) insured versus 50 (SD 14) uninsured; female 0.55 versus 0.52; income (\$1,000) 30 (SD 15) versus 32 (SD 25). Compute SMDs and variance ratios; flag problems.

<details><summary>Show solution</summary>

**Step 1.** Age: pooled SD $\sqrt{(144 + 196)/2} \approx 13.04$; $SMD \approx -0.38$ (flag). VR $\approx 0.73$.

**Step 2.** Female: SDs $\sqrt{0.2475}$, $\sqrt{0.2496}$; pooled ≈ 0.4985; $SMD \approx 0.06$ (fine).

**Step 3.** Income: pooled $\sqrt{(225 + 625)/2} \approx 20.6$; $SMD \approx -0.10$ (borderline); VR $= 225/625 = 0.36$ (flag: the uninsured have much more dispersed incomes).

**Answer:** fix age imbalance and the income variance mismatch (balance on income squared or quantiles).

</details>

*Your work:*

### Exercise 12 · KL between age distributions

Shares of treated and control patients in three age bands: treated (0.5, 0.3, 0.2), controls (0.2, 0.4, 0.4). Compute both KL divergences.

<details><summary>Show solution</summary>

**Step 1.** $D(T \,\|\, C) = 0.5\ln 2.5 + 0.3\ln 0.75 + 0.2\ln 0.5 = 0.458 - 0.086 - 0.139 = 0.233$.

**Step 2.** $D(C \,\|\, T) = 0.2\ln 0.4 + 0.4\ln(4/3) + 0.4\ln 2 = -0.183 + 0.115 + 0.277 = 0.209$.

**Answer:** different, as expected; either is a summary of how much reweighting is needed.

</details>

*Your work:*

### Exercise 13 · How unequal are these weights?

Normalised weights on five control clinics: 0.4, 0.3, 0.1, 0.1, 0.1. Compute ESS, $D_{KL}(w \,\|\, u)$ and the bound $\ln(n/ESS)$.

<details><summary>Show solution</summary>

**Step 1.** $\sum w^2 = 0.16 + 0.09 + 0.03 = 0.28$; $ESS = 3.57$.

**Step 2.** $D_{KL} = 0.4\ln 2 + 0.3\ln 1.5 + 0.3\ln 0.5 = 0.277 + 0.122 - 0.208 = 0.191$.

**Step 3.** Bound: $\ln(5/3.57) = \ln 1.4 = 0.336 \ge 0.191$.

</details>

*Your work:*

### Exercise 14 · Tails matter

After matching, mean household income is balanced (SMD 0.02) but the variance ratio is 3. The outcome (catastrophic health spending) is concentrated among the poorest households. What is the risk?

<details><summary>Show solution</summary>

**Step 1.** More dispersed incomes in one group mean more very poor households there.

**Step 2.** The outcome responds non-linearly to income, so the groups' expected outcomes differ even with equal means.

**Answer:** biased estimates; balance quantiles (or add income squared, or a poverty indicator) as well as the mean.

</details>

*Your work:*

### Exercise 15 · Balance on what matters

Explain why balancing a "prognostic score" (predicted untreated outcome from covariates) is a useful complement to covariate-by-covariate balance.

<details><summary>Show solution</summary>

**Step 1.** Bias from imbalance in covariate $X_k$ is roughly (imbalance) × (effect of $X_k$ on the outcome).

**Step 2.** A prognostic score summarises covariates weighted by their importance for the outcome (estimated on controls, or on a separate sample to avoid peeking).

**Answer:** small imbalance in the prognostic score indicates small bias, even if some unimportant covariates are imbalanced.

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np

smd = lambda m1, m0, s1, s0: (m1 - m0) / np.sqrt((s1 ** 2 + s0 ** 2) / 2)
H = lambda p: -(np.asarray(p) * np.log(p)).sum()
kl = lambda p, q: (np.asarray(p) * np.log(np.asarray(p) / np.asarray(q))).sum()
kln = lambda m1, s1, m2, s2: np.log(s2 / s1) + (s1 ** 2 + (m1 - m2) ** 2) / (2 * s2 ** 2) - .5
print("Ex 1 SMD:", round(smd(40, 35, 10, 8), 3), " VR:", round(100 / 64, 3))
print("Ex 3 entropy:", round(H([.5, .25, .25]), 4), " ln 3:", round(np.log(3), 4), " Ex 4 KL:", round(kl([.3, .7], [.5, .5]), 4))
print("Ex 5/8 normal KLs:", round(kln(1, 1, 0, 1), 3), round(kln(0, 2, 0, 1), 3), round(kln(0, 1, 0, 2), 3), " Ex 10:", round(kln(0, 3, 0, 1), 3))
print("Ex 11 SMDs:", round(smd(45, 50, 12, 14), 3), round(smd(.55, .52, np.sqrt(.55 * .45), np.sqrt(.52 * .48)), 3), round(smd(30, 32, 15, 25), 3), " VR income:", 225 / 625)
print("Ex 12:", round(kl([.5, .3, .2], [.2, .4, .4]), 4), round(kl([.2, .4, .4], [.5, .3, .2]), 4))
w = np.array([.4, .3, .1, .1, .1]); ess = 1 / (w ** 2).sum()
print("Ex 13: ESS", round(ess, 3), " KL", round(kl(w, np.full(5, .2)), 4), " ln n - H", round(np.log(5) - H(w), 4), " bound", round(np.log(5 / ess), 4))

# Monte Carlo check of the normal KL formula
rng = np.random.default_rng(83); x = rng.normal(1, 1, 1_000_000)
logp = lambda x, m, s: -np.log(s) - (x - m) ** 2 / (2 * s ** 2)
print("MC KL N(1,1)||N(0,1):", round((logp(x, 1, 1) - logp(x, 0, 1)).mean(), 4))

## Key takeaways

- Judge balance with standardised differences (|SMD| < 0.1), variance ratios and distributional checks, before and after adjustment.
- KL divergence measures how far one distribution is from another; it is non-negative and asymmetric, with closed forms for normals.
- For weights, KL from uniform equals ln n minus their entropy and is bounded by ln(n/ESS): a measure of how much information weighting throws away.